<a href="https://colab.research.google.com/github/Habibaaboalhassan66/swimming-detection/blob/main/FramesPreprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install opencv-python-headless numpy tqdm

In [ ]:

EVERY_N     = 3
SAVE_DEBUG  = False
VIDEO_EXTENSIONS = {".mp4", ".mov", ".avi", ".mkv", ".m4v", ".MP4", ".MOV"}

In [ ]:
"""
swimmer_preprocess_drive.py  v2  — RAM-safe edition
─────────────────────────────────────────────────────
KEY CHANGE FROM v1:
  v1 loaded ALL frames into a list before processing → crashed Colab RAM.
  v2 processes frame-by-frame in a single pass, holding only a small
  rolling window in memory at any time (~30 frames max).

What changed and why:
  • Removed 2-pass architecture   → single pass, frame-by-frame streaming
  • Removed all_frames list        → frames read, processed, saved, discarded
  • Added MAX_DIM resize on read   → frames capped at 720p before anything
                                     else; pose models don't need 4K input
  • Stabilizer uses rolling buffer → keeps last STAB_SMOOTH_RADIUS*2 frames
                                     of transforms instead of the full video
  • Added gc.collect() per video   → forces Python to release memory between
                                     videos so RAM doesn't accumulate across
                                     a long batch
  • All preprocessing logic        → identical to v1 (blur, CLAHE, reflect,
                                     shadow, ROI) — nothing changed there

HOW TO USE IN COLAB:
  Cell 1:  !pip install opencv-python-headless numpy tqdm
  Cell 2:  set OLD_FOLDER, NEW_FOLDER paths (config block below)
  Cell 3:  paste this entire file
"""

# ══════════════════════════════════════════════
#  ① MOUNT GOOGLE DRIVE
# ══════════════════════════════════════════════
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
print("✅ Google Drive mounted")

# ══════════════════════════════════════════════
#  ② USER CONFIG  ← edit these
# ══════════════════════════════════════════════

OLD_FOLDER  = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/Front view/old"    # ← CHANGE
NEW_FOLDER  = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/Front view/new"    # ← CHANGE
OUTPUT_ROOT = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames"

EVERY_N     = 3      # save every Nth frame  (3 = every 3rd)
SAVE_DEBUG  = False  # side-by-side debug images (uses 2x disk)
MAX_DIM     = 720    # resize frames so longest side ≤ this (set None to disable)

VIDEO_EXTENSIONS = {".mp4", ".mov", ".avi", ".mkv", ".m4v", ".MP4", ".MOV"}

# ══════════════════════════════════════════════
#  ③ IMPORTS
# ══════════════════════════════════════════════
import os, gc
import cv2
import numpy as np
from tqdm import tqdm
from pathlib import Path
from collections import deque
from datetime import datetime

# ── Tuneables ─────────────────────────────────
BLUR_THRESHOLD     = 80.0
SHARPEN_STRENGTH   = 1.5
CLAHE_CLIP         = 2.5
CLAHE_TILE         = (8, 8)
REFLECTION_THRESH  = 240
REFLECTION_DILATE  = 7
SHADOW_GAMMA       = 1.6
SHADOW_DARK_THRESH = 60
ROI_PADDING        = 40
STAB_SMOOTH_RADIUS = 15   # rolling window half-size for stabilization


# ══════════════════════════════════════════════
#  ④ FRAME RESIZE  (new in v2)
# ══════════════════════════════════════════════
def resize_frame(frame: np.ndarray, max_dim: int) -> np.ndarray:
    """
    Downscale so the longest side ≤ max_dim, preserving aspect ratio.
    No-op if the frame is already small enough.
    WHY: A 4K frame is 3840×2160 = ~25 MB per frame in RAM.
         At 720p it's ~2.8 MB — 9× less memory, same pose-estimation quality.
    """
    if max_dim is None:
        return frame
    h, w = frame.shape[:2]
    if max(h, w) <= max_dim:
        return frame
    scale = max_dim / max(h, w)
    new_w = int(w * scale)
    new_h = int(h * scale)
    return cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA)


# ══════════════════════════════════════════════
#  ⑤ ROLLING STABILIZER  (rewritten in v2)
# ══════════════════════════════════════════════
class RollingStabilizer:
    """
    Single-pass stabilizer using a rolling deque of recent transforms.

    v1 problem: stored ALL transforms then smoothed over the full array.
    v2 fix:     keeps only the last (2*radius+1) transforms in a deque.
                The smoothed value for the current frame is the mean of
                whatever is in the window — good enough for steady-cam
                correction without storing the entire video.

    Trade-off: smoothing near the start/end of the video is slightly less
    accurate than full-video smoothing, but RAM usage is O(radius) not O(N).
    """

    def __init__(self, smooth_radius: int = STAB_SMOOTH_RADIUS):
        self.radius      = smooth_radius
        self._window     = deque(maxlen=2 * smooth_radius + 1)
        self._prev_gray  = None
        self._prev_pts   = None
        self._cum_raw    = np.zeros(3)   # cumulative raw transform
        self._cum_smooth = np.zeros(3)   # cumulative smoothed transform

    def _detect_points(self, gray):
        return cv2.goodFeaturesToTrack(
            gray, maxCorners=200, qualityLevel=0.01,
            minDistance=30, blockSize=3,
        )

    def update_and_warp(self, frame: np.ndarray) -> np.ndarray:
        """
        Compute the incremental transform for this frame, smooth it over
        the rolling window, and return the stabilized frame.
        All in one call — no separate accumulate/warp steps.
        """
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        if self._prev_gray is None:
            self._prev_gray = gray
            self._prev_pts  = self._detect_points(gray)
            self._window.append(np.zeros(3))
            return frame

        if self._prev_pts is None or len(self._prev_pts) < 10:
            self._prev_pts = self._detect_points(self._prev_gray)

        curr_pts, status, _ = cv2.calcOpticalFlowPyrLK(
            self._prev_gray, gray, self._prev_pts, None
        )

        raw_transform = np.zeros(3)
        if curr_pts is not None and status is not None:
            idx       = status.ravel() == 1
            prev_good = self._prev_pts[idx]
            curr_good = curr_pts[idx]
            if len(prev_good) >= 4:
                m, _ = cv2.estimateAffinePartial2D(prev_good, curr_good)
                if m is not None:
                    raw_transform = np.array([
                        m[0, 2], m[1, 2], np.arctan2(m[1, 0], m[0, 0])
                    ])

        self._window.append(raw_transform)
        self._cum_raw    += raw_transform
        smoothed_step     = np.mean(self._window, axis=0)
        self._cum_smooth += smoothed_step
        diff              = self._cum_smooth - self._cum_raw

        self._prev_gray = gray
        self._prev_pts  = self._detect_points(gray)

        dx, dy, da = diff
        h, w = frame.shape[:2]
        M = np.array([
            [np.cos(da), -np.sin(da), dx],
            [np.sin(da),  np.cos(da), dy],
        ], dtype=np.float32)
        return cv2.warpAffine(frame, M, (w, h), borderMode=cv2.BORDER_REPLICATE)


# ══════════════════════════════════════════════
#  ⑥ PREPROCESSING STEPS  (unchanged from v1)
# ══════════════════════════════════════════════
def reduce_motion_blur(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    if cv2.Laplacian(gray, cv2.CV_64F).var() < BLUR_THRESHOLD:
        blurred = cv2.GaussianBlur(frame, (0, 0), 3)
        return cv2.addWeighted(frame, 1 + SHARPEN_STRENGTH,
                               blurred, -SHARPEN_STRENGTH, 0)
    return frame

def normalize_lighting(frame):
    lab     = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe   = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_TILE)
    lab_eq  = cv2.merge([clahe.apply(l), a, b])
    return cv2.cvtColor(lab_eq, cv2.COLOR_LAB2BGR)

def remove_reflections(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, REFLECTION_THRESH, 255, cv2.THRESH_BINARY)
    kernel  = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (REFLECTION_DILATE, REFLECTION_DILATE))
    mask = cv2.dilate(mask, kernel)
    if mask.sum() == 0:
        return frame
    return cv2.inpaint(frame, mask, inpaintRadius=5, flags=cv2.INPAINT_TELEA)

def adjust_shadows(frame):
    hsv     = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV).astype(np.float32)
    h, s, v = cv2.split(hsv)
    shadow  = v < SHADOW_DARK_THRESH
    v_gamma = np.power(np.clip(v / 255.0, 1e-6, 1.0), 1.0 / SHADOW_GAMMA) * 255.0
    v[shadow] = v_gamma[shadow]
    return cv2.cvtColor(
        cv2.merge([h, s, np.clip(v, 0, 255)]).astype(np.uint8),
        cv2.COLOR_HSV2BGR)

class SwimmerROI:
    def __init__(self):
        self.bg_sub     = cv2.createBackgroundSubtractorMOG2(
            history=120, varThreshold=40, detectShadows=True)
        self._last_bbox = None

    def apply(self, frame):
        fg = self.bg_sub.apply(frame)
        _, fg = cv2.threshold(fg, 200, 255, cv2.THRESH_BINARY)
        fg = cv2.morphologyEx(fg, cv2.MORPH_CLOSE,
            cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15)))
        fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN,
            cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
        contours, _ = cv2.findContours(fg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        h, w = frame.shape[:2]
        if contours:
            largest = max(contours, key=cv2.contourArea)
            if cv2.contourArea(largest) > 500:
                x, y, bw, bh = cv2.boundingRect(largest)
                p = ROI_PADDING
                self._last_bbox = (max(0,x-p), max(0,y-p),
                                   min(w,x+bw+p), min(h,y+bh+p))
        if self._last_bbox is None:
            return frame
        x1,y1,x2,y2 = self._last_bbox
        canvas = np.zeros_like(frame)
        canvas[y1:y2, x1:x2] = frame[y1:y2, x1:x2]
        return canvas


# ══════════════════════════════════════════════
#  ⑦ SINGLE-VIDEO PROCESSOR  (single-pass in v2)
# ══════════════════════════════════════════════
def process_video(video_info: dict, output_root: str,
                  every_n: int, save_debug: bool) -> dict:
    label      = video_info["label"]
    stem       = video_info["stem"]
    video_path = video_info["path"]
    out_dir    = os.path.join(output_root, label, stem)

    # Skip if already processed
    existing = list(Path(out_dir).glob("*.jpg")) if Path(out_dir).exists() else []
    if existing:
        print(f"\n  ⏭️  SKIP [{label}/{stem}] — {len(existing)} frames already exist")
        return {"video": stem, "label": label, "status": "skipped",
                "frames_saved": len(existing)}

    os.makedirs(out_dir, exist_ok=True)
    if save_debug:
        os.makedirs(os.path.join(out_dir, "debug"), exist_ok=True)

    print(f"\n  🎬 [{label}] {stem}")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"  ❌ Cannot open: {video_path}")
        return {"video": stem, "label": label, "status": "error", "frames_saved": 0}

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps   = cap.get(cv2.CAP_PROP_FPS)
    print(f"     {total} frames @ {fps:.0f}fps  |  saving every {every_n}  |  max_dim={MAX_DIM}")

    stabilizer = RollingStabilizer()
    # roi = SwimmerROI()  — disabled, full frame output
    saved      = 0

    for idx in tqdm(range(total), desc="     proc", unit="f", leave=False):
        ret, frame = cap.read()
        if not ret:
            break

        # ── v2: resize immediately after read ──
        frame = resize_frame(frame, MAX_DIM)

        # ── preprocessing pipeline ──
        frame = stabilizer.update_and_warp(frame)   # step 1 stabilize
        frame = reduce_motion_blur(frame)            # step 2 blur
        frame = normalize_lighting(frame)            # step 3 contrast
        frame = remove_reflections(frame)            # step 4 reflections
        frame = adjust_shadows(frame)                # step 5 shadows
        # ROI crop disabled — full frame kept (user preference)
        # frame = roi.apply(frame)

        if idx % every_n == 0:
            cv2.imwrite(
                os.path.join(out_dir, f"frame_{idx:06d}.jpg"),
                frame, [cv2.IMWRITE_JPEG_QUALITY, 95]
            )
            saved += 1

        # frame goes out of scope here → Python frees it

    cap.release()

    # Force memory release before next video
    del stabilizer
    gc.collect()

    print(f"     ✅ {saved} frames saved → {out_dir}")
    return {"video": stem, "label": label, "status": "done", "frames_saved": saved}


# ══════════════════════════════════════════════
#  ⑧ FOLDER SCANNER
# ══════════════════════════════════════════════
def find_videos(folder: str, label: str) -> list:
    p = Path(folder)
    if not p.exists():
        print(f"  ⚠️  [{label}] Not found: {folder}  ← check your path in the CONFIG block")
        return []
    videos = [{"path": str(f), "stem": f.stem, "label": label}
              for f in sorted(p.rglob("*")) if f.suffix in VIDEO_EXTENSIONS]
    print(f"  📁 [{label}] {len(videos)} video(s) in {folder}")
    return videos


# ══════════════════════════════════════════════
#  ⑨ RUN
# ══════════════════════════════════════════════
print("\n" + "═"*60)
print("  SWIMMER PREPROCESSING  v2  (RAM-safe single-pass)")
print("  Started:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("═"*60)

print("\n📂 Scanning folders …")
all_videos = find_videos(OLD_FOLDER, "old") + find_videos(NEW_FOLDER, "new")

if not all_videos:
    print("\n❌ No videos found. Update OLD_FOLDER / NEW_FOLDER above.")
else:
    print(f"\n▶  {len(all_videos)} video(s) to process  |  output → {OUTPUT_ROOT}\n")
    results = []
    for i, vid in enumerate(all_videos, 1):
        print(f"[{i}/{len(all_videos)}]", end="")
        results.append(process_video(vid, OUTPUT_ROOT, EVERY_N, SAVE_DEBUG))

    print("\n" + "═"*60)
    print("  SUMMARY")
    print("═"*60)
    print(f"  {'Video':<35} {'Type':<5} {'Status':<10} {'Frames':>8}")
    print(f"  {'-'*35} {'-'*5} {'-'*10} {'-'*8}")
    for r in results:
        print(f"  {r['video']:<35} {r['label']:<5} {r['status']:<10} {r['frames_saved']:>8,}")
    total_f = sum(r["frames_saved"] for r in results)
    done    = sum(1 for r in results if r["status"] == "done")
    skipped = sum(1 for r in results if r["status"] == "skipped")
    errors  = sum(1 for r in results if r["status"] == "error")
    print(f"\n  Done: {done}  Skipped: {skipped}  Errors: {errors}  |  Total frames: {total_f:,}")
    print("═"*60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted

════════════════════════════════════════════════════════════
  SWIMMER PREPROCESSING  v2  (RAM-safe single-pass)
  Started: 2026-05-25 16:25:12
════════════════════════════════════════════════════════════

📂 Scanning folders …
  📁 [old] 12 video(s) in /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/Front view/old
  📁 [new] 17 video(s) in /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/Front view/new

▶  29 video(s) to process  |  output → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames

[1/29]
  🎬 [old] breaststroke_front_S01
     630 frames @ 60fps  |  saving every 3  |  max_dim=720


     ✅ 210 frames saved → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames/old/breaststroke_front_S01
[2/29]
  🎬 [old] breaststroke_front_S02
     536 frames @ 60fps  |  saving every 3  |  max_dim=720


     ✅ 179 frames saved → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames/old/breaststroke_front_S02
[3/29]
  🎬 [old] breaststroke_front_S03
     636 frames @ 57fps  |  saving every 3  |  max_dim=720


     ✅ 203 frames saved → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames/old/breaststroke_front_S03
[4/29]
  🎬 [old] breaststroke_front_S04
     588 frames @ 60fps  |  saving every 3  |  max_dim=720


     ✅ 196 frames saved → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames/old/breaststroke_front_S04
[5/29]
  🎬 [old] butterfly_front_S01
     378 frames @ 60fps  |  saving every 3  |  max_dim=720


...

     ✅ 285 frames saved → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames/new/freestyle_front_S05
[26/29]
  🎬 [new] freestyle_front_S06
     856 frames @ 60fps  |  saving every 3  |  max_dim=720


     ✅ 286 frames saved → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames/new/freestyle_front_S06
[27/29]
  🎬 [new] freestyle_front_S07
     700 frames @ 60fps  |  saving every 3  |  max_dim=720


     ✅ 234 frames saved → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames/new/freestyle_front_S07
[28/29]
  🎬 [new] freestyle_front_S08
     599 frames @ 60fps  |  saving every 3  |  max_dim=720


     ✅ 200 frames saved → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames/new/freestyle_front_S08
[29/29]
  🎬 [new] freestyle_front_S09
     486 frames @ 60fps  |  saving every 3  |  max_dim=720


     ✅ 162 frames saved → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames/new/freestyle_front_S09

════════════════════════════════════════════════════════════
  SUMMARY
════════════════════════════════════════════════════════════
  Video                               Type  Status       Frames
  ----------------------------------- ----- ---------- --------
  breaststroke_front_S01              old   done            210
  breaststroke_front_S02              old   done            179
  breaststroke_front_S03              old   done            203
  breaststroke_front_S04              old   done            196
  butterfly_front_S01                 old   done            126
  butterfly_front_S02                 old   done            124
  butterfly_front_S03                 old   done            182
  butterfly_front_S04                 old   done            136
  freestyle_front_S01                 old   done            193
  freestyle_front_S02      

Upload frames from drive

```
# This is formatted as code
```



In [ ]:
from roboflow import Roboflow
from pathlib import Path
import json, os

# ── CONFIG ──────────────────────────────────────
API_KEY   = "2gU1ewQS0rfbADpO4tCb"
WORKSPACE = "habibas-workspace"
PROJECT   = "swimmer-breaststroke-front"

# Folder of frames to upload (change per stroke)
UPLOAD_FOLDER = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames"

# This file tracks what's already been uploaded
# Saved on Drive so it survives disconnects
TRACKER = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/uploaded_tracker.json"
# ─────────────────────────────────────────────────

# Load existing tracker
if os.path.exists(TRACKER):
    with open(TRACKER) as f:
        uploaded = set(json.load(f))
    print(f"📋 Tracker loaded — {len(uploaded)} already uploaded")
else:
    uploaded = set()
    print("📋 No tracker found — starting fresh")

# Connect to Roboflow
rf      = Roboflow(api_key=API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)

# Find all frames
all_images = sorted(Path(UPLOAD_FOLDER).rglob("*.jpg"))
remaining  = [f for f in all_images if str(f) not in uploaded]
print(f"📁 Total frames : {len(all_images)}")
print(f"⏭️  Already done : {len(uploaded)}")
print(f"▶️  To upload    : {len(remaining)}")

# Upload
for i, img in enumerate(remaining):
    try:
        project.upload(str(img))
        uploaded.add(str(img))

        # Save tracker every 50 uploads
        # So if disconnected you don't lose progress
        if i % 50 == 0:
            with open(TRACKER, "w") as f:
                json.dump(list(uploaded), f)
            print(f"  {i}/{len(remaining)} uploaded — progress saved")

    except Exception as e:
        print(f"  ⚠️ Failed: {img.name} — {e}")

# Final save
with open(TRACKER, "w") as f:
    json.dump(list(uploaded), f)

print(f"\n✅ Done — {len(uploaded)} total frames uploaded to Roboflow")

📋 No tracker found — starting fresh
loading Roboflow workspace...
loading Roboflow project...
📁 Total frames : 5781
⏭️  Already done : 0
▶️  To upload    : 5781
  0/5781 uploaded — progress saved
  50/5781 uploaded — progress saved
  100/5781 uploaded — progress saved


KeyboardInterrupt: 

In [ ]:
from pathlib import Path
root = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames"
for f in Path(root).iterdir():
    print(f.name)

freestyle
breaststroke
Butterfly


In [ ]:
API_KEY = "2gU1ewQS0rfbADpO4tCb"

In [ ]:
"""
roboflow_upload.py
───────────────────
Uploads frames from 3 stroke folders to their matching Roboflow projects.

HOW TO USE IN COLAB:
  Cell 1:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip install roboflow -q

  Cell 2:
    paste and run this entire file
"""

from roboflow import Roboflow
from pathlib import Path
import json, os

# ══════════════════════════════════════════════
#  CONFIG  ← only edit this block
# ══════════════════════════════════════════════
API_KEY   = "2gU1ewQS0rfbADpO4tCb"       # ← paste your Roboflow API key
WORKSPACE = "habibas-workspace"       # ← confirmed

BASE_FOLDER = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames"

TRACKER = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/uploaded_tracker.json"

# Stroke folder name → Roboflow project name
STROKE_PROJECTS = {
    "breaststroke": "swimmer-breaststroke-front",
    "freestyle":    "swimmer-freestyle-front",
    "Butterfly":    "swimmer-butterfly-front",
}
# ══════════════════════════════════════════════

# Load tracker
if os.path.exists(TRACKER):
    with open(TRACKER) as f:
        uploaded = set(json.load(f))
    print(f"📋 Tracker loaded — {len(uploaded)} already uploaded\n")
else:
    uploaded = set()
    print("📋 No tracker found — starting fresh\n")

# Connect to Roboflow
rf = Roboflow(api_key=API_KEY)

# Loop over each stroke
for stroke_folder, project_name in STROKE_PROJECTS.items():

    folder_path = Path(BASE_FOLDER) / stroke_folder

    print(f"{'═'*55}")
    print(f"  Stroke  : {stroke_folder}")
    print(f"  Project : {project_name}")
    print(f"  Folder  : {folder_path}")

    # Check folder exists
    if not folder_path.exists():
        print(f"  ⚠️  Folder not found — skipping\n")
        continue

    # Connect to this stroke's project
    try:
        project = rf.workspace(WORKSPACE).project(project_name)
    except Exception as e:
        print(f"  ❌ Could not connect to project: {e}\n")
        continue

    # Find all frames
    all_images = sorted(folder_path.rglob("*.jpg"))
    remaining  = [f for f in all_images if str(f) not in uploaded]

    print(f"  Total frames : {len(all_images)}")
    print(f"  Already done : {len(all_images) - len(remaining)}")
    print(f"  To upload    : {len(remaining)}\n")

    if not remaining:
        print(f"  ✅ All frames already uploaded — skipping\n")
        continue

    # Upload
    failed = 0
    for i, img in enumerate(remaining):
        try:
            project.upload(str(img))
            uploaded.add(str(img))

            # Save tracker every 50 uploads
            if i % 50 == 0:
                with open(TRACKER, "w") as f:
                    json.dump(list(uploaded), f)
                print(f"  [{stroke_folder}] {i}/{len(remaining)} uploaded")

        except Exception as e:
            failed += 1
            print(f"  ⚠️  Failed: {img.name} — {e}")

    # Save tracker after each stroke completes
    with open(TRACKER, "w") as f:
        json.dump(list(uploaded), f)

    print(f"\n  ✅ {stroke_folder} done — {len(remaining) - failed} uploaded, {failed} failed\n")

# Final summary
print(f"{'═'*55}")
print(f"  ALL DONE")
print(f"  Total frames uploaded across all strokes: {len(uploaded)}")
print(f"{'═'*55}")


📋 Tracker loaded — 101 already uploaded

═══════════════════════════════════════════════════════
  Stroke  : breaststroke
  Project : swimmer-breaststroke-front
  Folder  : /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames/breaststroke
loading Roboflow workspace...
loading Roboflow project...
  Total frames : 2188
  Already done : 0
  To upload    : 2188

  [breaststroke] 0/2188 uploaded
  [breaststroke] 50/2188 uploaded
  [breaststroke] 100/2188 uploaded
  [breaststroke] 150/2188 uploaded
  [breaststroke] 200/2188 uploaded
  [breaststroke] 250/2188 uploaded
  [breaststroke] 300/2188 uploaded
  [breaststroke] 350/2188 uploaded
  [breaststroke] 400/2188 uploaded
  [breaststroke] 450/2188 uploaded
  [breaststroke] 500/2188 uploaded
  [breaststroke] 550/2188 uploaded
  [breaststroke] 600/2188 uploaded
  [breaststroke] 650/2188 uploaded
  [breaststroke] 700/2188 uploaded
  [breaststroke] 750/2188 uploaded
  [breaststroke] 800/2188 uploaded
  [breaststrok

In [ ]:
import shutil
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
print("✅ Google Drive mounted")

shutil.rmtree("/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames")

print("✅ Deleted")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted
✅ Deleted


In [ ]:
"""
swimmer_preprocess_drive.py  v3  — crop top background
────────────────────────────────────────────────────────
What changed from v2:
  • Added CROP_TOP_RATIO = 0.40  → crops top 40% (sky/buildings) from every
    frame then resizes back to original height — effectively zooms in 1.67×
    on the water area where the swimmer is, without any detection logic.
  • Added crop_top() function in pipeline after resize_frame()
  • Everything else identical to v2

HOW TO USE IN COLAB:
  Cell 1:  !pip install opencv-python-headless numpy tqdm
  Cell 2:  set OLD_FOLDER, NEW_FOLDER paths (config block below)
  Cell 3:  paste this entire file
"""

# ══════════════════════════════════════════════
#  ① MOUNT GOOGLE DRIVE
# ══════════════════════════════════════════════
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
print("✅ Google Drive mounted")

# ══════════════════════════════════════════════
#  ② USER CONFIG  ← edit these
# ══════════════════════════════════════════════

OLD_FOLDER  = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/Front view/old"    # ← CHANGE
NEW_FOLDER  = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/Front view/new"    # ← CHANGE
OUTPUT_ROOT = "/content/drive/MyDrive/Swimming/processed_frames"

EVERY_N          = 3      # save every Nth frame  (3 = every 3rd)
SAVE_DEBUG       = False  # side-by-side debug images (uses 2x disk)
MAX_DIM          = 720    # resize frames so longest side ≤ this (set None to disable)
CROP_TOP_RATIO   = 0.40   # crop top 40% (sky + buildings) — set 0.0 to disable

VIDEO_EXTENSIONS = {".mp4", ".mov", ".avi", ".mkv", ".m4v", ".MP4", ".MOV"}

# ══════════════════════════════════════════════
#  ③ IMPORTS
# ══════════════════════════════════════════════
import os, gc
import cv2
import numpy as np
from tqdm import tqdm
from pathlib import Path
from collections import deque
from datetime import datetime

# ── Tuneables ─────────────────────────────────
BLUR_THRESHOLD     = 80.0
SHARPEN_STRENGTH   = 1.5
CLAHE_CLIP         = 2.5
CLAHE_TILE         = (8, 8)
REFLECTION_THRESH  = 240
REFLECTION_DILATE  = 7
SHADOW_GAMMA       = 1.6
SHADOW_DARK_THRESH = 60
ROI_PADDING        = 40
STAB_SMOOTH_RADIUS = 15   # rolling window half-size for stabilization


# ══════════════════════════════════════════════
#  ④ FRAME RESIZE  (new in v2)
# ══════════════════════════════════════════════
def resize_frame(frame: np.ndarray, max_dim: int) -> np.ndarray:
    """
    Downscale so the longest side ≤ max_dim, preserving aspect ratio.
    No-op if the frame is already small enough.
    WHY: A 4K frame is 3840×2160 = ~25 MB per frame in RAM.
         At 720p it's ~2.8 MB — 9× less memory, same pose-estimation quality.
    """
    if max_dim is None:
        return frame
    h, w = frame.shape[:2]
    if max(h, w) <= max_dim:
        return frame
    scale = max_dim / max(h, w)
    new_w = int(w * scale)
    new_h = int(h * scale)
    return cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA)


# ══════════════════════════════════════════════
#  NEW: CROP TOP  — remove sky/buildings, keep water area
# ══════════════════════════════════════════════
def crop_top(frame: np.ndarray, ratio: float) -> np.ndarray:
    """
    Removes the top `ratio` portion of the frame (sky, buildings, pool deck)
    and resizes the remaining water area back to the original height.

    WHY: Front-view cameras are positioned far from the pool, so the top
    40% of the frame is background. Cropping it and resizing back to full
    height effectively zooms in on the swimmer 1.67× without any detection.

    ratio = 0.40 → crop top 40%, keep bottom 60%, resize back to full height
    ratio = 0.0  → no crop (disable)
    """
    if ratio <= 0.0:
        return frame
    h, w    = frame.shape[:2]
    cut     = int(h * ratio)
    cropped = frame[cut:, :]                          # remove top portion
    return cv2.resize(cropped, (w, h),                # resize back to original dims
                      interpolation=cv2.INTER_LINEAR)


# ══════════════════════════════════════════════
#  ⑤ ROLLING STABILIZER  (rewritten in v2)
# ══════════════════════════════════════════════
class RollingStabilizer:
    """
    Single-pass stabilizer using a rolling deque of recent transforms.

    v1 problem: stored ALL transforms then smoothed over the full array.
    v2 fix:     keeps only the last (2*radius+1) transforms in a deque.
                The smoothed value for the current frame is the mean of
                whatever is in the window — good enough for steady-cam
                correction without storing the entire video.

    Trade-off: smoothing near the start/end of the video is slightly less
    accurate than full-video smoothing, but RAM usage is O(radius) not O(N).
    """

    def __init__(self, smooth_radius: int = STAB_SMOOTH_RADIUS):
        self.radius      = smooth_radius
        self._window     = deque(maxlen=2 * smooth_radius + 1)
        self._prev_gray  = None
        self._prev_pts   = None
        self._cum_raw    = np.zeros(3)   # cumulative raw transform
        self._cum_smooth = np.zeros(3)   # cumulative smoothed transform

    def _detect_points(self, gray):
        return cv2.goodFeaturesToTrack(
            gray, maxCorners=200, qualityLevel=0.01,
            minDistance=30, blockSize=3,
        )

    def update_and_warp(self, frame: np.ndarray) -> np.ndarray:
        """
        Compute the incremental transform for this frame, smooth it over
        the rolling window, and return the stabilized frame.
        All in one call — no separate accumulate/warp steps.
        """
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        if self._prev_gray is None:
            self._prev_gray = gray
            self._prev_pts  = self._detect_points(gray)
            self._window.append(np.zeros(3))
            return frame

        if self._prev_pts is None or len(self._prev_pts) < 10:
            self._prev_pts = self._detect_points(self._prev_gray)

        curr_pts, status, _ = cv2.calcOpticalFlowPyrLK(
            self._prev_gray, gray, self._prev_pts, None
        )

        raw_transform = np.zeros(3)
        if curr_pts is not None and status is not None:
            idx       = status.ravel() == 1
            prev_good = self._prev_pts[idx]
            curr_good = curr_pts[idx]
            if len(prev_good) >= 4:
                m, _ = cv2.estimateAffinePartial2D(prev_good, curr_good)
                if m is not None:
                    raw_transform = np.array([
                        m[0, 2], m[1, 2], np.arctan2(m[1, 0], m[0, 0])
                    ])

        self._window.append(raw_transform)
        self._cum_raw    += raw_transform
        smoothed_step     = np.mean(self._window, axis=0)
        self._cum_smooth += smoothed_step
        diff              = self._cum_smooth - self._cum_raw

        self._prev_gray = gray
        self._prev_pts  = self._detect_points(gray)

        dx, dy, da = diff
        h, w = frame.shape[:2]
        M = np.array([
            [np.cos(da), -np.sin(da), dx],
            [np.sin(da),  np.cos(da), dy],
        ], dtype=np.float32)
        return cv2.warpAffine(frame, M, (w, h), borderMode=cv2.BORDER_REPLICATE)


# ══════════════════════════════════════════════
#  ⑥ PREPROCESSING STEPS  (unchanged from v1)
# ══════════════════════════════════════════════
def reduce_motion_blur(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    if cv2.Laplacian(gray, cv2.CV_64F).var() < BLUR_THRESHOLD:
        blurred = cv2.GaussianBlur(frame, (0, 0), 3)
        return cv2.addWeighted(frame, 1 + SHARPEN_STRENGTH,
                               blurred, -SHARPEN_STRENGTH, 0)
    return frame

def normalize_lighting(frame):
    lab     = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe   = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_TILE)
    lab_eq  = cv2.merge([clahe.apply(l), a, b])
    return cv2.cvtColor(lab_eq, cv2.COLOR_LAB2BGR)

def remove_reflections(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, REFLECTION_THRESH, 255, cv2.THRESH_BINARY)
    kernel  = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (REFLECTION_DILATE, REFLECTION_DILATE))
    mask = cv2.dilate(mask, kernel)
    if mask.sum() == 0:
        return frame
    return cv2.inpaint(frame, mask, inpaintRadius=5, flags=cv2.INPAINT_TELEA)

def adjust_shadows(frame):
    hsv     = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV).astype(np.float32)
    h, s, v = cv2.split(hsv)
    shadow  = v < SHADOW_DARK_THRESH
    v_gamma = np.power(np.clip(v / 255.0, 1e-6, 1.0), 1.0 / SHADOW_GAMMA) * 255.0
    v[shadow] = v_gamma[shadow]
    return cv2.cvtColor(
        cv2.merge([h, s, np.clip(v, 0, 255)]).astype(np.uint8),
        cv2.COLOR_HSV2BGR)

class SwimmerROI:
    def __init__(self):
        self.bg_sub     = cv2.createBackgroundSubtractorMOG2(
            history=120, varThreshold=40, detectShadows=True)
        self._last_bbox = None

    def apply(self, frame):
        fg = self.bg_sub.apply(frame)
        _, fg = cv2.threshold(fg, 200, 255, cv2.THRESH_BINARY)
        fg = cv2.morphologyEx(fg, cv2.MORPH_CLOSE,
            cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15)))
        fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN,
            cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
        contours, _ = cv2.findContours(fg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        h, w = frame.shape[:2]
        if contours:
            largest = max(contours, key=cv2.contourArea)
            if cv2.contourArea(largest) > 500:
                x, y, bw, bh = cv2.boundingRect(largest)
                p = ROI_PADDING
                self._last_bbox = (max(0,x-p), max(0,y-p),
                                   min(w,x+bw+p), min(h,y+bh+p))
        if self._last_bbox is None:
            return frame
        x1,y1,x2,y2 = self._last_bbox
        canvas = np.zeros_like(frame)
        canvas[y1:y2, x1:x2] = frame[y1:y2, x1:x2]
        return canvas


# ══════════════════════════════════════════════
#  ⑦ SINGLE-VIDEO PROCESSOR  (single-pass in v2)
# ══════════════════════════════════════════════
def process_video(video_info: dict, output_root: str,
                  every_n: int, save_debug: bool) -> dict:
    label      = video_info["label"]
    stem       = video_info["stem"]
    video_path = video_info["path"]
    out_dir    = os.path.join(output_root, label, stem)

    # Skip if already processed
    existing = list(Path(out_dir).glob("*.jpg")) if Path(out_dir).exists() else []
    if existing:
        print(f"\n  ⏭️  SKIP [{label}/{stem}] — {len(existing)} frames already exist")
        return {"video": stem, "label": label, "status": "skipped",
                "frames_saved": len(existing)}

    os.makedirs(out_dir, exist_ok=True)
    if save_debug:
        os.makedirs(os.path.join(out_dir, "debug"), exist_ok=True)

    print(f"\n  🎬 [{label}] {stem}")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"  ❌ Cannot open: {video_path}")
        return {"video": stem, "label": label, "status": "error", "frames_saved": 0}

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps   = cap.get(cv2.CAP_PROP_FPS)
    print(f"     {total} frames @ {fps:.0f}fps  |  saving every {every_n}  |  max_dim={MAX_DIM}")

    stabilizer = RollingStabilizer()
    # roi = SwimmerROI()  — disabled, full frame output
    saved      = 0

    for idx in tqdm(range(total), desc="     proc", unit="f", leave=False):
        ret, frame = cap.read()
        if not ret:
            break

        # ── v2: resize immediately after read ──
        frame = resize_frame(frame, MAX_DIM)

        # ── crop top background (sky/buildings) ──
        frame = crop_top(frame, CROP_TOP_RATIO)

        # ── preprocessing pipeline ──
        frame = stabilizer.update_and_warp(frame)   # step 1 stabilize
        frame = reduce_motion_blur(frame)            # step 2 blur
        frame = normalize_lighting(frame)            # step 3 contrast
        frame = remove_reflections(frame)            # step 4 reflections
        frame = adjust_shadows(frame)                # step 5 shadows
        # ROI crop disabled — full frame kept (user preference)
        # frame = roi.apply(frame)

        if idx % every_n == 0:
            cv2.imwrite(
                os.path.join(out_dir, f"frame_{idx:06d}.jpg"),
                frame, [cv2.IMWRITE_JPEG_QUALITY, 95]
            )
            saved += 1

        # frame goes out of scope here → Python frees it

    cap.release()

    # Force memory release before next video
    del stabilizer
    gc.collect()

    print(f"     ✅ {saved} frames saved → {out_dir}")
    return {"video": stem, "label": label, "status": "done", "frames_saved": saved}


# ══════════════════════════════════════════════
#  ⑧ FOLDER SCANNER
# ══════════════════════════════════════════════
def find_videos(folder: str, label: str) -> list:
    p = Path(folder)
    if not p.exists():
        print(f"  ⚠️  [{label}] Not found: {folder}  ← check your path in the CONFIG block")
        return []
    videos = [{"path": str(f), "stem": f.stem, "label": label}
              for f in sorted(p.rglob("*")) if f.suffix in VIDEO_EXTENSIONS]
    print(f"  📁 [{label}] {len(videos)} video(s) in {folder}")
    return videos


# ══════════════════════════════════════════════
#  ⑨ RUN
# ══════════════════════════════════════════════
print("\n" + "═"*60)
print("  SWIMMER PREPROCESSING  v2  (RAM-safe single-pass)")
print("  Started:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("═"*60)

print("\n📂 Scanning folders …")
all_videos = find_videos(OLD_FOLDER, "old") + find_videos(NEW_FOLDER, "new")

if not all_videos:
    print("\n❌ No videos found. Update OLD_FOLDER / NEW_FOLDER above.")
else:
    print(f"\n▶  {len(all_videos)} video(s) to process  |  output → {OUTPUT_ROOT}\n")
    results = []
    for i, vid in enumerate(all_videos, 1):
        print(f"[{i}/{len(all_videos)}]", end="")
        results.append(process_video(vid, OUTPUT_ROOT, EVERY_N, SAVE_DEBUG))

    print("\n" + "═"*60)
    print("  SUMMARY")
    print("═"*60)
    print(f"  {'Video':<35} {'Type':<5} {'Status':<10} {'Frames':>8}")
    print(f"  {'-'*35} {'-'*5} {'-'*10} {'-'*8}")
    for r in results:
        print(f"  {r['video']:<35} {r['label']:<5} {r['status']:<10} {r['frames_saved']:>8,}")
    total_f = sum(r["frames_saved"] for r in results)
    done    = sum(1 for r in results if r["status"] == "done")
    skipped = sum(1 for r in results if r["status"] == "skipped")
    errors  = sum(1 for r in results if r["status"] == "error")
    print(f"\n  Done: {done}  Skipped: {skipped}  Errors: {errors}  |  Total frames: {total_f:,}")
    print("═"*60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted

════════════════════════════════════════════════════════════
  SWIMMER PREPROCESSING  v2  (RAM-safe single-pass)
  Started: 2026-05-28 14:03:09
════════════════════════════════════════════════════════════

📂 Scanning folders …
  📁 [old] 12 video(s) in /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/Front view/old
  📁 [new] 17 video(s) in /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/Front view/new

▶  29 video(s) to process  |  output → /content/drive/MyDrive/Swimming/processed_frames

[1/29]
  🎬 [old] breaststroke_front_S01
     630 frames @ 60fps  |  saving every 3  |  max_dim=720


     ✅ 210 frames saved → /content/drive/MyDrive/Swimming/processed_frames/old/breaststroke_front_S01
[2/29]
  🎬 [old] breaststroke_front_S02
     536 frames @ 60fps  |  saving every 3  |  max_dim=720


     ✅ 179 frames saved → /content/drive/MyDrive/Swimming/processed_frames/old/breaststroke_front_S02
[3/29]
  🎬 [old] breaststroke_front_S03
     636 frames @ 57fps  |  saving every 3  |  max_dim=720


     ✅ 203 frames saved → /content/drive/MyDrive/Swimming/processed_frames/old/breaststroke_front_S03
[4/29]
  🎬 [old] breaststroke_front_S04
     588 frames @ 60fps  |  saving every 3  |  max_dim=720


     ✅ 196 frames saved → /content/drive/MyDrive/Swimming/processed_frames/old/breaststroke_front_S04
[5/29]
  🎬 [old] butterfly_front_S01
     378 frames @ 60fps  |  saving every 3  |  max_dim=720


...

     ✅ 285 frames saved → /content/drive/MyDrive/Swimming/processed_frames/new/freestyle_front_S05
[26/29]
  🎬 [new] freestyle_front_S06
     856 frames @ 60fps  |  saving every 3  |  max_dim=720


     ✅ 286 frames saved → /content/drive/MyDrive/Swimming/processed_frames/new/freestyle_front_S06
[27/29]
  🎬 [new] freestyle_front_S07
     700 frames @ 60fps  |  saving every 3  |  max_dim=720


     ✅ 234 frames saved → /content/drive/MyDrive/Swimming/processed_frames/new/freestyle_front_S07
[28/29]
  🎬 [new] freestyle_front_S08
     599 frames @ 60fps  |  saving every 3  |  max_dim=720


     ✅ 200 frames saved → /content/drive/MyDrive/Swimming/processed_frames/new/freestyle_front_S08
[29/29]
  🎬 [new] freestyle_front_S09
     486 frames @ 60fps  |  saving every 3  |  max_dim=720


     ✅ 162 frames saved → /content/drive/MyDrive/Swimming/processed_frames/new/freestyle_front_S09

════════════════════════════════════════════════════════════
  SUMMARY
════════════════════════════════════════════════════════════
  Video                               Type  Status       Frames
  ----------------------------------- ----- ---------- --------
  breaststroke_front_S01              old   done            210
  breaststroke_front_S02              old   done            179
  breaststroke_front_S03              old   done            203
  breaststroke_front_S04              old   done            196
  butterfly_front_S01                 old   done            126
  butterfly_front_S02                 old   done            124
  butterfly_front_S03                 old   done            182
  butterfly_front_S04                 old   done            136
  freestyle_front_S01                 old   done            193
  freestyle_front_S02                 old   done            229


In [ ]:
"""
roboflow_upload.py
───────────────────
Uploads frames from 3 stroke folders to their matching Roboflow projects.

HOW TO USE IN COLAB:
  Cell 1:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip install roboflow -q

  Cell 2:
    paste and run this entire file
"""

from roboflow import Roboflow
from pathlib import Path
import json, os

# ══════════════════════════════════════════════
#  CONFIG  ← only edit this block
# ══════════════════════════════════════════════
API_KEY   = "2gU1ewQS0rfbADpO4tCb"       # ← paste your Roboflow API key
WORKSPACE = "habibas-workspace"       # ← confirmed

BASE_FOLDER = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames"

TRACKER = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/uploaded_tracker.json"

# Stroke folder name → Roboflow project name
STROKE_PROJECTS = {
    "breaststroke": "swimmer-breaststroke-front1",
    "freestyle":    "swimmer-freestyle-front1",
    "Butterfly":    "swimmer-butterfly-front1",
}
# ══════════════════════════════════════════════

# Load tracker
if os.path.exists(TRACKER):
    with open(TRACKER) as f:
        uploaded = set(json.load(f))
    print(f"📋 Tracker loaded — {len(uploaded)} already uploaded\n")
else:
    uploaded = set()
    print("📋 No tracker found — starting fresh\n")

# Connect to Roboflow
rf = Roboflow(api_key=API_KEY)

# Loop over each stroke
for stroke_folder, project_name in STROKE_PROJECTS.items():

    folder_path = Path(BASE_FOLDER) / stroke_folder

    print(f"{'═'*55}")
    print(f"  Stroke  : {stroke_folder}")
    print(f"  Project : {project_name}")
    print(f"  Folder  : {folder_path}")

    # Check folder exists
    if not folder_path.exists():
        print(f"  ⚠️  Folder not found — skipping\n")
        continue

    # Connect to this stroke's project
    try:
        project = rf.workspace(WORKSPACE).project(project_name)
    except Exception as e:
        print(f"  ❌ Could not connect to project: {e}\n")
        continue

    # Find all frames
    all_images = sorted(folder_path.rglob("*.jpg"))
    remaining  = [f for f in all_images if str(f) not in uploaded]

    print(f"  Total frames : {len(all_images)}")
    print(f"  Already done : {len(all_images) - len(remaining)}")
    print(f"  To upload    : {len(remaining)}\n")

    if not remaining:
        print(f"  ✅ All frames already uploaded — skipping\n")
        continue

    # Upload
    failed = 0
    for i, img in enumerate(remaining):
        try:
            project.upload(str(img))
            uploaded.add(str(img))

            # Save tracker every 50 uploads
            if i % 50 == 0:
                with open(TRACKER, "w") as f:
                    json.dump(list(uploaded), f)
                print(f"  [{stroke_folder}] {i}/{len(remaining)} uploaded")

        except Exception as e:
            failed += 1
            print(f"  ⚠️  Failed: {img.name} — {e}")

    # Save tracker after each stroke completes
    with open(TRACKER, "w") as f:
        json.dump(list(uploaded), f)

    print(f"\n  ✅ {stroke_folder} done — {len(remaining) - failed} uploaded, {failed} failed\n")

# Final summary
print(f"{'═'*55}")
print(f"  ALL DONE")
print(f"  Total frames uploaded across all strokes: {len(uploaded)}")
print(f"{'═'*55}")

📋 Tracker loaded — 852 already uploaded

═══════════════════════════════════════════════════════
  Stroke  : breaststroke
  Project : swimmer-breaststroke-front1
  Folder  : /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames/breaststroke
loading Roboflow workspace...
loading Roboflow project...
  Total frames : 2188
  Already done : 851
  To upload    : 1337

  [breaststroke] 0/1337 uploaded
  [breaststroke] 50/1337 uploaded
  [breaststroke] 100/1337 uploaded
  [breaststroke] 150/1337 uploaded
  [breaststroke] 200/1337 uploaded
  [breaststroke] 250/1337 uploaded
  [breaststroke] 300/1337 uploaded
  [breaststroke] 350/1337 uploaded
  [breaststroke] 400/1337 uploaded
  [breaststroke] 450/1337 uploaded
  [breaststroke] 500/1337 uploaded
  [breaststroke] 550/1337 uploaded
  [breaststroke] 600/1337 uploaded
  [breaststroke] 650/1337 uploaded
  [breaststroke] 700/1337 uploaded
  [breaststroke] 750/1337 uploaded
  [breaststroke] 800/1337 uploaded
  [breastst

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!pip install roboflow -q
print("✅ Ready")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Ready


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
print(os.path.exists("/content/drive/MyDrive"))

Mounted at /content/drive
True


In [ ]:
"""
swimmer_preprocess_v4.py
─────────────────────────
Clean preprocessing — NO zoom, NO black borders, NO cropping.
Full frame is kept exactly as-is in terms of content and size.

Changes from v3:
  • Removed CROP_TOP_RATIO     → no top crop, full frame kept
  • Removed SwimmerROI         → no background blacking out
  • Removed center crop        → no zoom of any kind
  • Stabilization kept         → fixes camera shake only
  • Blur reduction kept        → sharpens only when needed
  • CLAHE contrast kept        → fixes lighting
  • Reflection removal kept    → fixes sun glare
  • Shadow adjustment kept     → fixes dark areas
  • Output is same aspect ratio as input, resized to 720p max

HOW TO USE IN COLAB:
  Cell 1:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip install opencv-python-headless numpy tqdm -q

  Cell 2:
    paste and run this entire file
"""

# ══════════════════════════════════════════════
#  ① MOUNT GOOGLE DRIVE
# ══════════════════════════════════════════════
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
print("✅ Google Drive mounted")

# ══════════════════════════════════════════════
#  ② USER CONFIG  ← edit these
# ══════════════════════════════════════════════
OLD_FOLDER  = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/Front view/old"
NEW_FOLDER  = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/Front view/new"
OUTPUT_ROOT = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames_v4"

EVERY_N    = 3      # save every Nth frame
SAVE_DEBUG = False  # side-by-side debug images
MAX_DIM    = 720    # resize so longest side = 720px, keep aspect ratio

VIDEO_EXTENSIONS = {".mp4", ".mov", ".avi", ".mkv", ".m4v", ".MP4", ".MOV"}

# ── Tuneables ─────────────────────────────────
BLUR_THRESHOLD     = 80.0   # Laplacian variance; below = sharpen
SHARPEN_STRENGTH   = 1.5    # unsharp mask strength
CLAHE_CLIP         = 2.5    # CLAHE clip limit
CLAHE_TILE         = (8, 8) # CLAHE tile grid
REFLECTION_THRESH  = 240    # brightness threshold for glare
REFLECTION_DILATE  = 7      # glare mask dilation kernel
SHADOW_GAMMA       = 1.6    # gamma lift for dark pixels
SHADOW_DARK_THRESH = 60     # HSV-V below this = shadow
STAB_SMOOTH_RADIUS = 15     # stabilization rolling window half-size


# ══════════════════════════════════════════════
#  ③ IMPORTS
# ══════════════════════════════════════════════
import os, gc
import cv2
import numpy as np
from tqdm import tqdm
from pathlib import Path
from collections import deque
from datetime import datetime


# ══════════════════════════════════════════════
#  ④ RESIZE  (aspect-ratio safe, no stretch)
# ══════════════════════════════════════════════
def resize_frame(frame: np.ndarray, max_dim: int) -> np.ndarray:
    """
    Resize so the longest side = max_dim.
    Preserves aspect ratio exactly — no stretching, no padding, no black bars.
    If frame is already smaller than max_dim, returns it unchanged.
    """
    if max_dim is None:
        return frame
    h, w = frame.shape[:2]
    if max(h, w) <= max_dim:
        return frame
    scale = max_dim / max(h, w)
    return cv2.resize(frame,
                      (int(w * scale), int(h * scale)),
                      interpolation=cv2.INTER_AREA)


# ══════════════════════════════════════════════
#  ⑤ ROLLING STABILIZER
# ══════════════════════════════════════════════
class RollingStabilizer:
    """
    Single-pass stabilizer using a rolling deque.
    Corrects camera shake only — does NOT move/crop/zoom content.
    Border pixels revealed by the warp are filled with BORDER_REPLICATE
    (nearest edge pixel) so there are NO black borders.
    """
    def __init__(self, smooth_radius: int = STAB_SMOOTH_RADIUS):
        self.radius      = smooth_radius
        self._window     = deque(maxlen=2 * smooth_radius + 1)
        self._prev_gray  = None
        self._prev_pts   = None
        self._cum_raw    = np.zeros(3)
        self._cum_smooth = np.zeros(3)

    def _detect_points(self, gray):
        return cv2.goodFeaturesToTrack(
            gray, maxCorners=200, qualityLevel=0.01,
            minDistance=30, blockSize=3)

    def update_and_warp(self, frame: np.ndarray) -> np.ndarray:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        if self._prev_gray is None:
            self._prev_gray = gray
            self._prev_pts  = self._detect_points(gray)
            self._window.append(np.zeros(3))
            return frame

        if self._prev_pts is None or len(self._prev_pts) < 10:
            self._prev_pts = self._detect_points(self._prev_gray)

        curr_pts, status, _ = cv2.calcOpticalFlowPyrLK(
            self._prev_gray, gray, self._prev_pts, None)

        raw_transform = np.zeros(3)
        if curr_pts is not None and status is not None:
            idx       = status.ravel() == 1
            prev_good = self._prev_pts[idx]
            curr_good = curr_pts[idx]
            if len(prev_good) >= 4:
                m, _ = cv2.estimateAffinePartial2D(prev_good, curr_good)
                if m is not None:
                    raw_transform = np.array([
                        m[0, 2], m[1, 2], np.arctan2(m[1, 0], m[0, 0])
                    ])

        self._window.append(raw_transform)
        self._cum_raw    += raw_transform
        smoothed_step     = np.mean(self._window, axis=0)
        self._cum_smooth += smoothed_step
        diff              = self._cum_smooth - self._cum_raw

        self._prev_gray = gray
        self._prev_pts  = self._detect_points(gray)

        dx, dy, da = diff
        h, w = frame.shape[:2]
        M = np.array([
            [np.cos(da), -np.sin(da), dx],
            [np.sin(da),  np.cos(da), dy],
        ], dtype=np.float32)

        # BORDER_REPLICATE → fills edge pixels with nearest real pixel
        # This guarantees NO black borders from stabilization
        return cv2.warpAffine(frame, M, (w, h),
                              borderMode=cv2.BORDER_REPLICATE)


# ══════════════════════════════════════════════
#  ⑥ PREPROCESSING STEPS
# ══════════════════════════════════════════════
def reduce_motion_blur(frame: np.ndarray) -> np.ndarray:
    """Sharpen only if Laplacian variance is below threshold."""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    if cv2.Laplacian(gray, cv2.CV_64F).var() < BLUR_THRESHOLD:
        blurred = cv2.GaussianBlur(frame, (0, 0), 3)
        return cv2.addWeighted(frame, 1 + SHARPEN_STRENGTH,
                               blurred, -SHARPEN_STRENGTH, 0)
    return frame


def normalize_lighting(frame: np.ndarray) -> np.ndarray:
    """CLAHE on LAB L-channel — fixes uneven pool lighting, keeps colors."""
    lab     = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe   = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_TILE)
    lab_eq  = cv2.merge([clahe.apply(l), a, b])
    return cv2.cvtColor(lab_eq, cv2.COLOR_LAB2BGR)


def remove_reflections(frame: np.ndarray) -> np.ndarray:
    """Inpaint bright specular highlights (sun glare on water)."""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, REFLECTION_THRESH, 255, cv2.THRESH_BINARY)
    kernel  = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (REFLECTION_DILATE, REFLECTION_DILATE))
    mask = cv2.dilate(mask, kernel)
    if mask.sum() == 0:
        return frame
    return cv2.inpaint(frame, mask, inpaintRadius=5, flags=cv2.INPAINT_TELEA)


def adjust_shadows(frame: np.ndarray) -> np.ndarray:
    """Gamma-lift only dark (shadowed) pixels in HSV V-channel."""
    hsv     = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV).astype(np.float32)
    h, s, v = cv2.split(hsv)
    shadow  = v < SHADOW_DARK_THRESH
    v_gamma = np.power(np.clip(v / 255.0, 1e-6, 1.0),
                       1.0 / SHADOW_GAMMA) * 255.0
    v[shadow] = v_gamma[shadow]
    return cv2.cvtColor(
        cv2.merge([h, s, np.clip(v, 0, 255)]).astype(np.uint8),
        cv2.COLOR_HSV2BGR)


# ══════════════════════════════════════════════
#  ⑦ SINGLE-VIDEO PROCESSOR
# ══════════════════════════════════════════════
def process_video(video_info: dict, output_root: str,
                  every_n: int, save_debug: bool) -> dict:
    label      = video_info["label"]
    stem       = video_info["stem"]
    video_path = video_info["path"]
    out_dir    = os.path.join(output_root, label, stem)

    # Skip if already processed
    existing = list(Path(out_dir).glob("*.jpg")) if Path(out_dir).exists() else []
    if existing:
        print(f"\n  ⏭️  SKIP [{label}/{stem}] — {len(existing)} frames exist")
        return {"video": stem, "label": label,
                "status": "skipped", "frames_saved": len(existing)}

    os.makedirs(out_dir, exist_ok=True)
    if save_debug:
        os.makedirs(os.path.join(out_dir, "debug"), exist_ok=True)

    print(f"\n  🎬 [{label}] {stem}")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"  ❌ Cannot open: {video_path}")
        return {"video": stem, "label": label,
                "status": "error", "frames_saved": 0}

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps   = cap.get(cv2.CAP_PROP_FPS)
    print(f"     {total} frames @ {fps:.0f}fps | every_n={every_n} | max_dim={MAX_DIM}")

    stabilizer = RollingStabilizer()
    saved = 0

    for idx in tqdm(range(total), desc="     proc", unit="f", leave=False):
        ret, frame = cap.read()
        if not ret:
            break

        # ── pipeline (NO crop, NO zoom, NO black) ──
        frame = resize_frame(frame, MAX_DIM)     # 1. resize (aspect-safe)
        frame = stabilizer.update_and_warp(frame) # 2. stabilize (replicate border)
        frame = reduce_motion_blur(frame)          # 3. sharpen if blurry
        frame = normalize_lighting(frame)          # 4. CLAHE contrast
        frame = remove_reflections(frame)          # 5. remove sun glare
        frame = adjust_shadows(frame)              # 6. lift shadows

        if idx % every_n == 0:
            out_path = os.path.join(out_dir, f"frame_{idx:06d}.jpg")
            cv2.imwrite(out_path, frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
            saved += 1

            if save_debug and idx > 0:
                # Read original again for comparison — only in debug mode
                pass  # skip to avoid re-reading; debug can be enabled separately

    cap.release()
    del stabilizer
    gc.collect()

    print(f"     ✅ {saved} frames → {out_dir}")
    return {"video": stem, "label": label,
            "status": "done", "frames_saved": saved}


# ══════════════════════════════════════════════
#  ⑧ FOLDER SCANNER
# ══════════════════════════════════════════════
def find_videos(folder: str, label: str) -> list:
    p = Path(folder)
    if not p.exists():
        print(f"  ⚠️  [{label}] Not found: {folder}")
        return []
    videos = [{"path": str(f), "stem": f.stem, "label": label}
              for f in sorted(p.rglob("*"))
              if f.suffix in VIDEO_EXTENSIONS]
    print(f"  📁 [{label}] {len(videos)} video(s)")
    return videos


# ══════════════════════════════════════════════
#  ⑨ RUN
# ══════════════════════════════════════════════
print("\n" + "═"*60)
print("  SWIMMER PREPROCESSING  v4  (no zoom, no black, no crop)")
print("  Started:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("═"*60)

print("\n📂 Scanning folders …")
all_videos = find_videos(OLD_FOLDER, "old") + find_videos(NEW_FOLDER, "new")

if not all_videos:
    print("\n❌ No videos found. Check OLD_FOLDER / NEW_FOLDER paths.")
else:
    print(f"\n▶  {len(all_videos)} video(s) | output → {OUTPUT_ROOT}\n")
    results = []
    for i, vid in enumerate(all_videos, 1):
        print(f"[{i}/{len(all_videos)}]", end="")
        results.append(process_video(vid, OUTPUT_ROOT, EVERY_N, SAVE_DEBUG))

    print("\n" + "═"*60)
    print("  SUMMARY")
    print("═"*60)
    print(f"  {'Video':<35} {'Type':<5} {'Status':<10} {'Frames':>8}")
    print(f"  {'-'*35} {'-'*5} {'-'*10} {'-'*8}")
    for r in results:
        print(f"  {r['video']:<35} {r['label']:<5} {r['status']:<10} {r['frames_saved']:>8,}")

    total_f = sum(r["frames_saved"] for r in results)
    done    = sum(1 for r in results if r["status"] == "done")
    skipped = sum(1 for r in results if r["status"] == "skipped")
    errors  = sum(1 for r in results if r["status"] == "error")
    print(f"\n  Done: {done} | Skipped: {skipped} | Errors: {errors} | Total frames: {total_f:,}")
    print("═"*60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted

════════════════════════════════════════════════════════════
  SWIMMER PREPROCESSING  v4  (no zoom, no black, no crop)
  Started: 2026-05-28 20:24:25
════════════════════════════════════════════════════════════

📂 Scanning folders …
  📁 [old] 12 video(s)
  📁 [new] 17 video(s)

▶  29 video(s) | output → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames_v4

[1/29]
  ⏭️  SKIP [old/breaststroke_front_S01] — 135 frames exist
[2/29]
  🎬 [old] breaststroke_front_S02
     536 frames @ 60fps | every_n=3 | max_dim=720


     ✅ 179 frames → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames_v4/old/breaststroke_front_S02
[3/29]
  🎬 [old] breaststroke_front_S03
     636 frames @ 57fps | every_n=3 | max_dim=720


     ✅ 203 frames → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames_v4/old/breaststroke_front_S03
[4/29]
  🎬 [old] breaststroke_front_S04
     588 frames @ 60fps | every_n=3 | max_dim=720


     ✅ 196 frames → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames_v4/old/breaststroke_front_S04
[5/29]
  🎬 [old] butterfly_front_S01
     378 frames @ 60fps | every_n=3 | max_dim=720


     ✅ 126 frames → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames_v4/old/butterfly_front_S01
[6/29]
  🎬 [old] butterfly_front_S02
     370 frames @ 60fps | every_n=3 | max_dim=720


...

     ✅ 285 frames → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames_v4/new/freestyle_front_S05
[26/29]
  🎬 [new] freestyle_front_S06
     856 frames @ 60fps | every_n=3 | max_dim=720


     ✅ 286 frames → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames_v4/new/freestyle_front_S06
[27/29]
  🎬 [new] freestyle_front_S07
     700 frames @ 60fps | every_n=3 | max_dim=720


     ✅ 234 frames → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames_v4/new/freestyle_front_S07
[28/29]
  🎬 [new] freestyle_front_S08
     599 frames @ 60fps | every_n=3 | max_dim=720


     ✅ 200 frames → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames_v4/new/freestyle_front_S08
[29/29]
  🎬 [new] freestyle_front_S09
     486 frames @ 60fps | every_n=3 | max_dim=720


     ✅ 162 frames → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames_v4/new/freestyle_front_S09

════════════════════════════════════════════════════════════
  SUMMARY
════════════════════════════════════════════════════════════
  Video                               Type  Status       Frames
  ----------------------------------- ----- ---------- --------
  breaststroke_front_S01              old   skipped         135
  breaststroke_front_S02              old   done            179
  breaststroke_front_S03              old   done            203
  breaststroke_front_S04              old   done            196
  butterfly_front_S01                 old   done            126
  butterfly_front_S02                 old   done            124
  butterfly_front_S03                 old   done            182
  butterfly_front_S04                 old   done            136
  freestyle_front_S01                 old   done            193
  freestyle_front_S02         

**Importing frames to Roboflow**

In [ ]:

!pip install roboflow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 82.2 MB/s eta 0:00:00


In [ ]:
from roboflow import Roboflow
from pathlib import Path
import json, os

API_KEY   = "2gU1ewQS0rfbADpO4tCb"        # ← paste your key
WORKSPACE = "habibas-workspace"

BASE_FOLDER = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames_v4"
TRACKER     = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/uploaded_tracker_v4.json"

# Each subfolder → its Roboflow project
STROKE_PROJECTS = {
    "old/breaststroke_front_S01":  "swimmer-breaststroke-front1",
    "old/breaststroke_front_S02":  "swimmer-breaststroke-front1",
    "old/breaststroke_front_S03":  "swimmer-breaststroke-front1",
    "old/breaststroke_front_S04":  "swimmer-breaststroke-front1",
    "new/breaststroke_front_S05":  "swimmer-breaststroke-front1",
    "new/breaststroke_front_S06":  "swimmer-breaststroke-front1",
    "new/breaststroke_front_S07":  "swimmer-breaststroke-front1",
    "new/breaststroke_front_S08":  "swimmer-breaststroke-front1",
    "new/breaststroke_front_S09":  "swimmer-breaststroke-front1",
    "new/breaststroke_front_S010": "swimmer-breaststroke-front1",
    "old/butterfly_front_S01":     "swimmer-butterfly-front1",
    "old/butterfly_front_S02":     "swimmer-butterfly-front1",
    "old/butterfly_front_S03":     "swimmer-butterfly-front1",
    "old/butterfly_front_S04":     "swimmer-butterfly-front1",
    "new/butterfly_front_S05":     "swimmer-butterfly-front1",
    "new/butterfly_front_S06":     "swimmer-butterfly-front1",
    "new/butterfly_front_S07":     "swimmer-butterfly-front1",
    "new/butterfly_front_S08":     "swimmer-butterfly-front1",
    "new/butterfly_front_S09":     "swimmer-butterfly-front1",
    "old/freestyle_front_S01":     "swimmer-freestyle-front1",
    "old/freestyle_front_S02":     "swimmer-freestyle-front1",
    "old/freestyle_front_S03":     "swimmer-freestyle-front1",
    "old/freestyle_front_S04":     "swimmer-freestyle-front1",
    "new/freestyle_front_S05":     "swimmer-freestyle-front1",
    "new/freestyle_front_S06":     "swimmer-freestyle-front1",
    "new/freestyle_front_S07":     "swimmer-freestyle-front1",
    "new/freestyle_front_S08":     "swimmer-freestyle-front1",
    "new/freestyle_front_S09":     "swimmer-freestyle-front1",
    "new/freestyle_front_S010":    "swimmer-freestyle-front1",
}

# Load tracker
if os.path.exists(TRACKER):
    with open(TRACKER) as f:
        uploaded = set(json.load(f))
    print(f"📋 {len(uploaded)} already uploaded")
else:
    uploaded = set()
    print("📋 Starting fresh")

rf = Roboflow(api_key=API_KEY)
connected = {}

for subfolder, project_name in STROKE_PROJECTS.items():
    folder_path = Path(BASE_FOLDER) / subfolder

    if not folder_path.exists():
        print(f"⚠️  Not found: {subfolder}")
        continue

    if project_name not in connected:
        connected[project_name] = rf.workspace(WORKSPACE).project(project_name)
    project = connected[project_name]

    images    = sorted(folder_path.glob("*.jpg"))
    remaining = [f for f in images if str(f) not in uploaded]

    print(f"\n📂 {subfolder} → {project_name} | {len(remaining)} to upload")

    failed = 0
    for i, img in enumerate(remaining):
        try:
            project.upload(str(img))
            uploaded.add(str(img))
            if i % 50 == 0:
                with open(TRACKER, "w") as f:
                    json.dump(list(uploaded), f)
                print(f"  {i}/{len(remaining)} uploaded")
        except Exception as e:
            failed += 1
            print(f"  ⚠️ Failed: {img.name} — {e}")

    with open(TRACKER, "w") as f:
        json.dump(list(uploaded), f)
    print(f"  ✅ Done — {len(remaining)-failed} uploaded")

print(f"\n{'═'*50}")
print(f"✅ All done — {len(uploaded)} total frames uploaded")
print(f"{'═'*50}")

📋 Starting fresh
loading Roboflow workspace...
loading Roboflow project...

📂 old/breaststroke_front_S01 → swimmer-breaststroke-front1 | 135 to upload
  0/135 uploaded
  50/135 uploaded
  100/135 uploaded
  ✅ Done — 135 uploaded

📂 old/breaststroke_front_S02 → swimmer-breaststroke-front1 | 179 to upload
  0/179 uploaded
  50/179 uploaded
  100/179 uploaded
  150/179 uploaded
  ✅ Done — 179 uploaded

📂 old/breaststroke_front_S03 → swimmer-breaststroke-front1 | 203 to upload
  0/203 uploaded
  50/203 uploaded
  100/203 uploaded
  150/203 uploaded
  200/203 uploaded
  ✅ Done — 203 uploaded

📂 old/breaststroke_front_S04 → swimmer-breaststroke-front1 | 196 to upload
  0/196 uploaded
  50/196 uploaded
  100/196 uploaded
  150/196 uploaded
  ✅ Done — 196 uploaded

📂 new/breaststroke_front_S05 → swimmer-breaststroke-front1 | 308 to upload
  0/308 uploaded
  50/308 uploaded
  100/308 uploaded
  150/308 uploaded
  200/308 uploaded
  250/308 uploaded
  300/308 uploaded
  ✅ Done — 308 uploaded

📂 